In [ ]:
import pandas as pd

#is the shared start of the three web addresses, so we don't repeat it.
base = "https://huggingface.co/datasets/masakhane/masakhaner2/resolve/refs%2Fconvert%2Fparquet/twi"

train_df = pd.read_parquet(f"{base}/train/0000.parquet")
val_df   = pd.read_parquet(f"{base}/validation/0000.parquet")
test_df  = pd.read_parquet(f"{base}/test/0000.parquet")

print(len(train_df), len(val_df), len(test_df))
train_df.head()


In [ ]:
# Tool that counts how often each item appears
from collections import Counter

# Put all words from all training sentences into one list
train_tokens = [word for sentence in train_df["tokens"] for word in sentence]

# Count how many times each word appears
word_counts = Counter(train_tokens)

# Show total words, distinct words, and words seen only once
print("Total tokens:", len(train_tokens))
print("Vocabulary size (types):", len(word_counts))
print("Words seen only once:", sum(1 for w, c in word_counts.items() if c == 1))
print()

# Show the 20 most common words
print("20 most common words:")
print(word_counts.most_common(20))


In [ ]:
# Tool for drawing charts
import matplotlib.pyplot as plt

# Word frequencies sorted from most to least common
frequencies = [count for word, count in word_counts.most_common()]

# Ranks: 1 for the most common word, 2 for the next, and so on
ranks = range(1, len(frequencies) + 1)

# Plot rank against frequency on log-log axes
plt.figure(figsize=(7, 5))
plt.loglog(ranks, frequencies)
plt.xlabel("Rank of word (log scale)")
plt.ylabel("Frequency (log scale)")
plt.title("Zipf's Law in Twi Training Data")
plt.grid(True)
plt.show()


In [ ]:
# Test Zipf: if rank and frequency are inversely proportional, rank × frequency stays roughly constant
for rank in [1, 2, 5, 10, 50, 100, 500, 1000, 5000]:
    word, freq = word_counts.most_common()[rank - 1]
    print(f"Rank {rank:>5} | {word:>12} | freq {freq:>5} | rank × freq = {rank * freq}")


In [ ]:
# Lowercase every word and add start/end markers to each sentence
def preprocess(sentences):
    cleaned = []
    for sentence in sentences:
        words = [word.lower() for word in sentence if any(ch.isalnum() for ch in word)]
        cleaned.append(["<s>"] + words + ["</s>"])
    return cleaned

# Apply the same cleaning to all three splits
train_sents = preprocess(train_df["tokens"])
val_sents   = preprocess(val_df["tokens"])
test_sents  = preprocess(test_df["tokens"])

# Compare a sentence before and after cleaning
print("Before:", list(train_df["tokens"][0]))
print("After: ", train_sents[0])


In [ ]:
# Count words in the cleaned training data
train_counts = Counter(word for sentence in train_sents for word in sentence)

# Vocabulary = words seen at least 2 times in training
vocab = {word for word, count in train_counts.items() if count >= 2}

# Replace any word not in the vocabulary with <UNK>
def replace_unk(sentences, vocab):
    return [[word if word in vocab else "<UNK>" for word in sentence] for sentence in sentences]

# Apply to all three splits, always using the TRAINING vocabulary
train_sents = replace_unk(train_sents, vocab)
val_sents   = replace_unk(val_sents, vocab)
test_sents  = replace_unk(test_sents, vocab)

# Check the results
print("Vocabulary size (incl. <UNK>):", len(vocab) + 1)
test_words = [word for sentence in test_sents for word in sentence]
print("Share of test words that are <UNK>:", round(test_words.count("<UNK>") / len(test_words) * 100, 1), "%")
print("Example:", train_sents[0])


In [ ]:
# Cut a sentence into N-grams: slide a window of n words along it
def get_ngrams(sentence, n):
    return [tuple(sentence[i:i + n]) for i in range(len(sentence) - n + 1)]

# Try it on a short example
example = ["<s>", "kofi", "kɔɔ", "fie", "</s>"]
print("Unigrams:", get_ngrams(example, 1))
print("Bigrams: ", get_ngrams(example, 2))
print("Trigrams:", get_ngrams(example, 3))


In [ ]:
# Count all unigrams, bigrams and trigrams in the training data
unigram_counts = Counter(ng for s in train_sents for ng in get_ngrams(s, 1))
bigram_counts  = Counter(ng for s in train_sents for ng in get_ngrams(s, 2))
trigram_counts = Counter(ng for s in train_sents for ng in get_ngrams(s, 3))

# How many different N-grams of each size did we see?
print("Unique unigrams:", len(unigram_counts))
print("Unique bigrams: ", len(bigram_counts))
print("Unique trigrams:", len(trigram_counts))
print()

# The most common bigrams
print("10 most common bigrams:")
for bigram, count in bigram_counts.most_common(10):
    print(bigram, count)


In [ ]:
# MLE bigram probability: P(word | prev) = count(prev, word) / count(prev)
def bigram_prob(prev, word):
    return bigram_counts[(prev, word)] / unigram_counts[(prev,)]

# Trying a few examples
print("P(mu | no)    =", round(bigram_prob("no", "mu"), 4))
print("P(</s> | no)  =", round(bigram_prob("no", "</s>"), 4))
print("P(a | wɔn)    =", round(bigram_prob("wɔn", "a"), 4))
print("P(kofi | no)  =", round(bigram_prob("no", "kofi"), 4))


P(mu | no)    = 0.0927
P(</s> | no)  = 0.0705
P(a | wɔn)    = 0.1567
P(kofi | no)  = 0.0


In [12]:
# Vocabulary size V (all unique words, including <s>, </s> and <UNK>)
V = len(unigram_counts)

# Laplace (add-one) bigram probability
def laplace_prob(prev, word):
    return (bigram_counts[(prev, word)] + 1) / (unigram_counts[(prev,)] + V)

# Compare MLE and Laplace on the same examples
for prev, word in [("no", "mu"), ("no", "</s>"), ("wɔn", "a"), ("no", "kofi")]:
    print(f"P({word} | {prev}):  MLE = {bigram_prob(prev, word):.4f}   Laplace = {laplace_prob(prev, word):.4f}")


P(mu | no):  MLE = 0.0927   Laplace = 0.0537
P(</s> | no):  MLE = 0.0705   Laplace = 0.0409
P(a | wɔn):  MLE = 0.1567   Laplace = 0.0429
P(kofi | no):  MLE = 0.0000   Laplace = 0.0001
